# BirdCLEF+ 2026 — Week 1: EDA + Baseline + Submission

**Goal:** Understand the data, build a working EfficientNet-B0 baseline on mel spectrograms, and produce a valid Kaggle submission.

**Sections:**
1. Setup & Imports
2. Configuration
3. EDA — Dataset Exploration
4. Audio → Mel Spectrogram Pipeline
5. Dataset & DataLoader
6. Model Architecture
7. Training Loop
8. Validation & OOF AUC
9. Inference & Submission

> **Competition metric:** ROC-AUC (class-mean average). Higher is better. No threshold needed at submission time.

## 1. Setup & Imports

In [ ]:
# Install any missing packages (run once)
# !pip install timm librosa audiomentations --quiet

In [ ]:
import os
import gc
import warnings
import random
import math
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

# Audio
import librosa
import librosa.display
import soundfile as sf

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 2. Configuration

All hyperparameters in one place — change here, nothing else needs editing.

In [ ]:
class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────
    BASE_DIR        = Path('/kaggle/input/birdclef-2026')
    TRAIN_AUDIO_DIR = BASE_DIR / 'train_audio'
    TEST_AUDIO_DIR  = BASE_DIR / 'test_soundscapes'
    TRAIN_CSV       = BASE_DIR / 'train_metadata.csv'
    TAXONOMY_CSV    = BASE_DIR / 'taxonomy.csv'
    SAMPLE_SUB      = BASE_DIR / 'sample_submission.csv'
    OUTPUT_DIR      = Path('/kaggle/working')

    # ── Audio ──────────────────────────────────────────────────────────────
    SAMPLE_RATE     = 32000          # Hz — competition standard
    WINDOW_SIZE     = 5              # seconds per training chunk
    AUDIO_LEN       = SAMPLE_RATE * WINDOW_SIZE  # samples per chunk

    # ── Mel spectrogram ────────────────────────────────────────────────────
    N_FFT           = 1024
    HOP_LENGTH      = 64             # short hop → fine time resolution
    N_MELS          = 136
    FMIN            = 20
    FMAX            = 16000
    TARGET_SHAPE    = (256, 256)     # resize before model input

    # ── Model ──────────────────────────────────────────────────────────────
    MODEL_NAME      = 'efficientnet_b0'   # from timm
    IN_CHANNELS     = 1                   # mono spectrogram
    PRETRAINED      = True

    # ── Training ───────────────────────────────────────────────────────────
    N_FOLDS         = 5
    TRAIN_FOLDS     = [0]            # Week 1: train fold 0 only (fast)
    EPOCHS          = 10             # Week 1 quick baseline
    BATCH_SIZE      = 32
    NUM_WORKERS     = 2
    LR              = 1e-3
    WEIGHT_DECAY    = 1e-4
    WARMUP_EPOCHS   = 1
    MIN_LR          = 1e-6
    MIXUP_ALPHA     = 0.15           # 0 = disabled, 0.15 = light

    # ── Inference ──────────────────────────────────────────────────────────
    INFER_BATCH     = 16
    INFER_WINDOW    = 5              # seconds
    INFER_STRIDE    = 5              # non-overlapping (faster for week 1)

    # ── Misc ───────────────────────────────────────────────────────────────
    SEED            = 42
    DEBUG           = False          # set True to run on 200 samples only

cfg = CFG()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded. Output dir:', cfg.OUTPUT_DIR)

## 3. EDA — Dataset Exploration

In [ ]:
# Load metadata
train_df = pd.read_csv(cfg.TRAIN_CSV)
print('Train shape:', train_df.shape)
print('Columns:', train_df.columns.tolist())
train_df.head(3)

In [ ]:
# Taxonomy: maps species code → common name, family, order
taxonomy_df = pd.read_csv(cfg.TAXONOMY_CSV)
print('Taxonomy shape:', taxonomy_df.shape)
print('Columns:', taxonomy_df.columns.tolist())
taxonomy_df.head(3)

In [ ]:
# ── How many species? ──────────────────────────────────────────────────────
species_col = 'primary_label' if 'primary_label' in train_df.columns else 'species_code'
all_classes = sorted(train_df[species_col].unique())
NUM_CLASSES = len(all_classes)
print(f'Total species (classes): {NUM_CLASSES}')

# Label encoder: species code → integer index
le = LabelEncoder()
le.fit(all_classes)
train_df['label_idx'] = le.transform(train_df[species_col])

# Save class list for inference
class_list = le.classes_.tolist()
print('First 10 classes:', class_list[:10])

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
counts = train_df[species_col].value_counts()
print(f'Min samples per class : {counts.min()}')
print(f'Max samples per class : {counts.max()}')
print(f'Median               : {counts.median()}')
print(f'Classes with < 5 samples: {(counts < 5).sum()}')
print(f'Classes with < 20 samples: {(counts < 20).sum()}')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Full distribution histogram
axes[0].hist(counts.values, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Samples per class')
axes[0].set_ylabel('Number of classes')
axes[0].set_title('Class imbalance distribution')
axes[0].axvline(counts.median(), color='red', linestyle='--', label=f'Median = {counts.median():.0f}')
axes[0].legend()

# Top-30 and bottom-30
top30 = counts.head(30)
axes[1].barh(range(30), top30.values, color='steelblue')
axes[1].set_yticks(range(30))
axes[1].set_yticklabels(top30.index, fontsize=7)
axes[1].set_xlabel('Sample count')
axes[1].set_title('Top 30 most frequent species')

plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Audio duration distribution ────────────────────────────────────────────
# We'll sample 500 files to estimate duration without loading all audio
sample_df = train_df.sample(min(500, len(train_df)), random_state=42)
durations = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Checking durations'):
    filepath = cfg.TRAIN_AUDIO_DIR / row['filename']
    if filepath.exists():
        try:
            info = sf.info(str(filepath))
            durations.append(info.duration)
        except Exception:
            pass

durations = np.array(durations)
print(f'Duration stats (seconds):')
print(f'  Min: {durations.min():.1f}s')
print(f'  Max: {durations.max():.1f}s')
print(f'  Mean: {durations.mean():.1f}s')
print(f'  Median: {np.median(durations):.1f}s')
print(f'  Clips < 5s: {(durations < 5).sum()} ({100*(durations<5).mean():.1f}%)')

plt.figure(figsize=(8, 3))
plt.hist(durations, bins=40, color='steelblue', edgecolor='white')
plt.axvline(5, color='red', linestyle='--', label='5s window')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.title('Training clip duration distribution (500-file sample)')
plt.legend()
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'eda_audio_duration.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualise mel spectrograms for 9 random species ────────────────────────
def load_audio_chunk(filepath: str, sr: int = 32000, duration: float = 5.0,
                     offset: float = 0.0) -> np.ndarray:
    """Load audio clip, pad/trim to exact duration."""
    target_len = int(sr * duration)
    try:
        y, _ = librosa.load(filepath, sr=sr, offset=offset, duration=duration, mono=True)
    except Exception:
        return np.zeros(target_len, dtype=np.float32)
    # Trim or pad
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]
    return y.astype(np.float32)


def audio_to_melspec(y: np.ndarray, cfg: CFG) -> np.ndarray:
    """Convert waveform → log-mel spectrogram (H x W float32)."""
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=cfg.SAMPLE_RATE,
        n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS,
        fmin=cfg.FMIN,
        fmax=cfg.FMAX,
        power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Normalise to [0, 1]
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    return mel_db.astype(np.float32)


# Sample one file per species (9 species for the grid)
sample_species = train_df.groupby(species_col).first().sample(9, random_state=42)

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, (sp_code, row) in zip(axes.flat, sample_species.iterrows()):
    filepath = str(cfg.TRAIN_AUDIO_DIR / row['filename'])
    y = load_audio_chunk(filepath, sr=cfg.SAMPLE_RATE, duration=5.0)
    spec = audio_to_melspec(y, cfg)
    ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(sp_code, fontsize=8)
    ax.axis('off')

plt.suptitle('Log-Mel Spectrograms — 9 random Pantanal species (5s clips)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'eda_mel_spectrograms.png', dpi=120, bbox_inches='tight')
plt.show()
print('Spectrogram shape:', spec.shape, '  (n_mels x time_frames)')

## 4. Audio → Mel Spectrogram Pipeline

All preprocessing functions used by both training and inference.

In [ ]:
import cv2  # for resize — install with: pip install opencv-python-headless

def resize_spectrogram(spec: np.ndarray, target_shape: Tuple[int,int]) -> np.ndarray:
    """Resize (H, W) spectrogram to target_shape using bicubic interpolation."""
    return cv2.resize(spec, (target_shape[1], target_shape[0]),
                      interpolation=cv2.INTER_CUBIC)


def spec_augment(spec: np.ndarray, 
                 freq_mask_max: int = 20,
                 time_mask_max: int = 40,
                 n_freq_masks: int = 2,
                 n_time_masks: int = 2) -> np.ndarray:
    """SpecAugment: random frequency and time masking."""
    spec = spec.copy()
    H, W = spec.shape
    for _ in range(n_freq_masks):
        f = random.randint(0, freq_mask_max)
        f0 = random.randint(0, max(0, H - f))
        spec[f0:f0 + f, :] = 0.0
    for _ in range(n_time_masks):
        t = random.randint(0, time_mask_max)
        t0 = random.randint(0, max(0, W - t))
        spec[:, t0:t0 + t] = 0.0
    return spec


def mixup_data(x: torch.Tensor, y: torch.Tensor, 
               alpha: float = 0.15) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Mixup augmentation for spectrograms."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    idx = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixed loss for mixup training."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def process_audio_to_tensor(filepath: str, cfg: CFG,
                            offset: float = 0.0,
                            augment: bool = False) -> torch.Tensor:
    """Full pipeline: file → float32 tensor (1, H, W) ready for model."""
    y = load_audio_chunk(filepath, sr=cfg.SAMPLE_RATE, 
                         duration=cfg.WINDOW_SIZE, offset=offset)
    spec = audio_to_melspec(y, cfg)
    if augment:
        spec = spec_augment(spec)
    spec = resize_spectrogram(spec, cfg.TARGET_SHAPE)
    tensor = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)  # (1, H, W)
    return tensor


# Quick sanity check
sample_file = str(cfg.TRAIN_AUDIO_DIR / train_df['filename'].iloc[0])
t = process_audio_to_tensor(sample_file, cfg, augment=True)
print('Tensor shape:', t.shape)  # should be (1, 256, 256)
print('Min / Max:', t.min().item(), '/', t.max().item())

## 5. Dataset & DataLoader

In [ ]:
class BirdCLEFDataset(Dataset):
    """
    Training dataset.
    Each item is a random 5s chunk from a training clip.
    For short clips (< 5s), the audio is zero-padded.
    """
    def __init__(self, df: pd.DataFrame, cfg: CFG, augment: bool = True):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.augment = augment
        self.audio_dir = cfg.TRAIN_AUDIO_DIR

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        filepath = str(self.audio_dir / row['filename'])

        # Random offset for data augmentation — see different parts of the clip
        try:
            info = sf.info(filepath)
            total_dur = info.duration
        except Exception:
            total_dur = self.cfg.WINDOW_SIZE

        max_offset = max(0.0, total_dur - self.cfg.WINDOW_SIZE)
        offset = random.uniform(0, max_offset) if self.augment and max_offset > 0 else 0.0

        tensor = process_audio_to_tensor(filepath, self.cfg,
                                         offset=offset,
                                         augment=self.augment)

        # One-hot label vector (multilabel-ready)
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        label[int(row['label_idx'])] = 1.0

        # Handle secondary labels if present
        if 'secondary_labels' in row and isinstance(row['secondary_labels'], str):
            secondary = row['secondary_labels'].strip("[]").replace("'", "").split()
            for sec_sp in secondary:
                sec_sp = sec_sp.strip().rstrip(',')
                if sec_sp in le.classes_:
                    sec_idx = le.transform([sec_sp])[0]
                    label[sec_idx] = 0.5  # soft label for background calls

        return tensor, label


class BirdCLEFInferenceDataset(Dataset):
    """
    Inference dataset for test soundscapes.
    Splits each 1-min file into non-overlapping 5s windows.
    Returns tensor + row_id for submission matching.
    """
    def __init__(self, filepaths: List[str], cfg: CFG):
        self.samples = []  # list of (filepath, offset, row_id)
        self.cfg = cfg
        for fp in filepaths:
            stem = Path(fp).stem  # e.g. 'XC123456'
            try:
                info = sf.info(fp)
                total_dur = info.duration
            except Exception:
                total_dur = 60.0
            offsets = np.arange(0, total_dur, cfg.INFER_STRIDE)
            for off in offsets:
                end_sec = int(off + cfg.INFER_WINDOW)
                row_id = f'{stem}_{end_sec}'
                self.samples.append((fp, float(off), row_id))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str]:
        fp, offset, row_id = self.samples[idx]
        tensor = process_audio_to_tensor(fp, self.cfg, offset=offset, augment=False)
        return tensor, row_id


print('Dataset classes defined.')

## 6. Model Architecture

EfficientNet-B0 with a custom 1-channel input (mono spectrogram) and multilabel head.

In [ ]:
class GeM(nn.Module):
    """Generalised Mean Pooling — better than global average for fine-grained audio."""
    def __init__(self, p: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), 1
        ).pow(1.0 / self.p)


class BirdCLEFModel(nn.Module):
    """
    EfficientNet-B0 backbone with:
    - 1-channel input (mono mel spectrogram)
    - GeM pooling
    - Dropout + BN head
    - Multilabel output (sigmoid at inference)
    """
    def __init__(self, num_classes: int, cfg: CFG):
        super().__init__()
        # Load pretrained backbone
        self.backbone = timm.create_model(
            cfg.MODEL_NAME,
            pretrained=cfg.PRETRAINED,
            num_classes=0,          # remove classifier
            global_pool='',         # we'll pool ourselves
            in_chans=1              # mono spectrogram
        )
        feat_dim = self.backbone.num_features

        self.pool = GeM()
        self.bn   = nn.BatchNorm1d(feat_dim)
        self.drop = nn.Dropout(p=0.3)
        self.fc   = nn.Linear(feat_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 1, H, W)
        feat = self.backbone(x)        # (B, C, h, w)
        feat = self.pool(feat)         # (B, C, 1, 1)
        feat = feat.flatten(1)         # (B, C)
        feat = self.bn(feat)
        feat = self.drop(feat)
        out  = self.fc(feat)           # (B, num_classes) — raw logits
        return out


# Test forward pass
model_test = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
dummy = torch.randn(2, 1, 256, 256).to(DEVICE)
out = model_test(dummy)
print('Model output shape:', out.shape)  # (2, NUM_CLASSES)
total_params = sum(p.numel() for p in model_test.parameters()) / 1e6
print(f'Parameters: {total_params:.1f}M')
del model_test, dummy, out
gc.collect()

## 7. Training Loop

In [ ]:
def get_scheduler(optimizer, cfg: CFG, steps_per_epoch: int):
    """Cosine annealing with linear warmup."""
    total_steps = cfg.EPOCHS * steps_per_epoch
    warmup_steps = cfg.WARMUP_EPOCHS * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        min_factor = cfg.MIN_LR / cfg.LR
        return max(min_factor, cosine)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def compute_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Class-mean ROC-AUC (competition metric).
    Skips classes with no positive examples.
    """
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            try:
                auc = roc_auc_score(y_true[:, i], y_pred[:, i])
                aucs.append(auc)
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else 0.0


def train_one_epoch(model, loader, optimizer, scheduler, criterion, cfg, epoch):
    model.train()
    losses = []
    loop = tqdm(loader, desc=f'  Train E{epoch}', leave=False)

    for x, y in loop:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # Mixup augmentation
        if cfg.MIXUP_ALPHA > 0 and random.random() > 0.5:
            x, y_a, y_b, lam = mixup_data(x, y, cfg.MIXUP_ALPHA)
            logits = model(x)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        else:
            logits = model(x)
            loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())
        loop.set_postfix(loss=f'{np.mean(losses[-20:]):.4f}',
                         lr=f'{scheduler.get_last_lr()[0]:.2e}')

    return np.mean(losses)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for x, y in tqdm(loader, desc='  Valid', leave=False):
        x = x.to(DEVICE)
        logits = model(x)
        preds = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y.numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    auc = compute_auc((all_labels > 0.5).astype(int), all_preds)
    return auc, all_preds, all_labels


print('Training utilities defined.')

In [ ]:
# ── Cross-validation split ─────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['label_idx'])):
    train_df.loc[val_idx, 'fold'] = fold

print('Fold distribution:')
print(train_df['fold'].value_counts().sort_index())

# DEBUG mode: tiny subset
if cfg.DEBUG:
    train_df = train_df.groupby(species_col).head(2).reset_index(drop=True)
    print(f'DEBUG mode: using {len(train_df)} samples')

In [ ]:
# ── Main training loop ─────────────────────────────────────────────────────
oof_preds = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
best_fold_aucs = {}

for fold in cfg.TRAIN_FOLDS:
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}')
    print(f'{"="*55}')

    trn_idx = train_df[train_df['fold'] != fold].index
    val_idx = train_df[train_df['fold'] == fold].index

    trn_df = train_df.loc[trn_idx].reset_index(drop=True)
    val_df = train_df.loc[val_idx].reset_index(drop=True)
    print(f'  Train: {len(trn_df)} | Val: {len(val_df)}')

    trn_dataset = BirdCLEFDataset(trn_df, cfg, augment=True)
    val_dataset = BirdCLEFDataset(val_df, cfg, augment=False)

    trn_loader = DataLoader(trn_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=True, num_workers=cfg.NUM_WORKERS,
                            pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=cfg.NUM_WORKERS,
                            pin_memory=True)

    # Model, loss, optimiser
    model = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.LR,
                                  weight_decay=cfg.WEIGHT_DECAY)
    scheduler = get_scheduler(optimizer, cfg, len(trn_loader))

    best_auc = 0.0
    best_path = cfg.OUTPUT_DIR / f'model_fold{fold}.pt'
    history = []

    for epoch in range(1, cfg.EPOCHS + 1):
        trn_loss = train_one_epoch(model, trn_loader, optimizer,
                                   scheduler, criterion, cfg, epoch)
        val_auc, preds, labels = validate(model, val_loader)

        history.append({'epoch': epoch, 'loss': trn_loss, 'val_auc': val_auc})
        lr_now = scheduler.get_last_lr()[0]
        print(f'  Epoch {epoch:02d}  loss={trn_loss:.4f}  val_AUC={val_auc:.4f}  lr={lr_now:.2e}')

        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), best_path)
            print(f'  ✔ Saved best model  (AUC={best_auc:.4f})')

    best_fold_aucs[fold] = best_auc
    print(f'\n  Fold {fold} best val AUC: {best_auc:.4f}')

    # Store OOF predictions using best model
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    _, oof_pred, oof_label = validate(model, val_loader)
    oof_preds[val_idx] = oof_pred
    oof_labels[val_idx] = (oof_label > 0.5).astype(np.float32)

    # Plot training curve
    hist_df = pd.DataFrame(history)
    fig, ax1 = plt.subplots(figsize=(8, 3))
    ax1.plot(hist_df['epoch'], hist_df['loss'], 'b-o', ms=4, label='Train loss')
    ax1.set_ylabel('BCE Loss', color='b')
    ax2 = ax1.twinx()
    ax2.plot(hist_df['epoch'], hist_df['val_auc'], 'r-s', ms=4, label='Val AUC')
    ax2.set_ylabel('Val ROC-AUC', color='r')
    ax1.set_xlabel('Epoch')
    plt.title(f'Fold {fold} Training Curve')
    fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / f'training_curve_fold{fold}.png', dpi=120, bbox_inches='tight')
    plt.show()

    del model, trn_loader, val_loader, trn_dataset, val_dataset
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nAll folds done. Best AUCs: {best_fold_aucs}')

## 8. OOF Validation Summary

In [ ]:
# OOF AUC for trained folds only
trained_val_idx = train_df[train_df['fold'].isin(cfg.TRAIN_FOLDS)].index
oof_auc = compute_auc(oof_labels[trained_val_idx], oof_preds[trained_val_idx])
print(f'OOF ROC-AUC (fold {cfg.TRAIN_FOLDS}): {oof_auc:.4f}')

# Per-class AUC histogram
per_class_auc = []
for i in range(NUM_CLASSES):
    true_i = oof_labels[trained_val_idx, i]
    pred_i = oof_preds[trained_val_idx, i]
    if true_i.sum() > 0:
        try:
            per_class_auc.append(roc_auc_score(true_i, pred_i))
        except Exception:
            pass

plt.figure(figsize=(8, 3))
plt.hist(per_class_auc, bins=30, color='steelblue', edgecolor='white')
plt.axvline(np.mean(per_class_auc), color='red', linestyle='--',
            label=f'Mean = {np.mean(per_class_auc):.3f}')
plt.xlabel('Per-class ROC-AUC')
plt.ylabel('Count')
plt.title('Per-class AUC distribution (OOF)')
plt.legend()
plt.tight_layout()
plt.savefig(cfg.OUTPUT_DIR / 'oof_per_class_auc.png', dpi=120, bbox_inches='tight')
plt.show()

# Classes with low AUC — these need attention in Week 2
low_auc_threshold = 0.6
print(f'\nClasses with AUC < {low_auc_threshold}: {sum(a < low_auc_threshold for a in per_class_auc)}')
print('These are your hardest classes — prioritize in Week 2 (pseudo labels / more augmentation)')

## 9. Inference & Submission

Run the trained model on test soundscapes and produce `submission.csv`.

In [ ]:
@torch.no_grad()
def run_inference(model_paths: List[str], test_filepaths: List[str],
                  cfg: CFG) -> pd.DataFrame:
    """
    Run inference with one or more models and average predictions.
    Returns a DataFrame with row_id as index and species columns.
    """
    infer_dataset = BirdCLEFInferenceDataset(test_filepaths, cfg)
    infer_loader  = DataLoader(infer_dataset, batch_size=cfg.INFER_BATCH,
                               shuffle=False, num_workers=cfg.NUM_WORKERS,
                               pin_memory=True)

    all_row_ids = [s[2] for s in infer_dataset.samples]
    ensemble_preds = np.zeros((len(infer_dataset), NUM_CLASSES), dtype=np.float32)

    for model_path in model_paths:
        print(f'  Running: {Path(model_path).name}')
        model = BirdCLEFModel(NUM_CLASSES, cfg).to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.eval()

        fold_preds = []
        for x, _ in tqdm(infer_loader, desc='  Inference', leave=False):
            x = x.to(DEVICE)
            logits = model(x)
            preds = torch.sigmoid(logits).cpu().numpy()
            fold_preds.append(preds)

        fold_preds = np.concatenate(fold_preds, axis=0)
        ensemble_preds += fold_preds / len(model_paths)

        del model
        gc.collect()
        torch.cuda.empty_cache()

    # Build submission DataFrame
    sub_df = pd.DataFrame(ensemble_preds, columns=class_list)
    sub_df.insert(0, 'row_id', all_row_ids)
    return sub_df


# ── Load sample submission to get required row_ids and columns ─────────────
sample_sub = pd.read_csv(cfg.SAMPLE_SUB)
print('Sample submission shape:', sample_sub.shape)
print('Required columns:', sample_sub.columns[:5].tolist(), '...')
print('Total rows:', len(sample_sub))

In [ ]:
# ── Find test soundscapes ──────────────────────────────────────────────────
test_files = sorted(cfg.TEST_AUDIO_DIR.glob('*.ogg'))
if not test_files:
    test_files = sorted(cfg.TEST_AUDIO_DIR.glob('*.wav'))
print(f'Found {len(test_files)} test soundscape files')

# ── Collect trained model weights ─────────────────────────────────────────
model_paths = [str(cfg.OUTPUT_DIR / f'model_fold{f}.pt') for f in cfg.TRAIN_FOLDS
               if (cfg.OUTPUT_DIR / f'model_fold{f}.pt').exists()]
print(f'Using {len(model_paths)} model(s) for inference')

# ── Run inference ──────────────────────────────────────────────────────────
if test_files and model_paths:
    pred_df = run_inference(model_paths, [str(f) for f in test_files], cfg)
    print('Prediction shape:', pred_df.shape)

    # ── Align with sample submission ───────────────────────────────────────
    # sample_sub defines the exact row_ids and column order required
    submission = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')

    # Fill any missing row_ids with a small constant (e.g. very rare species)
    species_cols = [c for c in submission.columns if c != 'row_id']
    submission[species_cols] = submission[species_cols].fillna(0.01)

    # Ensure column order matches sample submission
    final_cols = sample_sub.columns.tolist()
    for col in final_cols:
        if col not in submission.columns and col != 'row_id':
            submission[col] = 0.01
    submission = submission[final_cols]

    sub_path = cfg.OUTPUT_DIR / 'submission.csv'
    submission.to_csv(sub_path, index=False)
    print(f'\nSubmission saved → {sub_path}')
    print(f'Shape: {submission.shape}')
    print(submission.head(3))
else:
    print('No test files or models found — skipping inference.')
    print('Run training first, then re-run this cell.')

## Submission Sanity Checks

Run these before hitting Submit to avoid a wasted daily submission.

In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────
try:
    sub = pd.read_csv(cfg.OUTPUT_DIR / 'submission.csv')
    ref = pd.read_csv(cfg.SAMPLE_SUB)

    checks = [
        ('Row count matches sample submission',
         len(sub) == len(ref)),
        ('Column count matches sample submission',
         len(sub.columns) == len(ref.columns)),
        ('No NaN values',
         sub.isnull().sum().sum() == 0),
        ('All species values in [0, 1]',
         (sub.drop('row_id', axis=1).values >= 0).all() and
         (sub.drop('row_id', axis=1).values <= 1).all()),
        ('row_id column is first',
         sub.columns[0] == 'row_id'),
        ('Column names match',
         list(sub.columns) == list(ref.columns)),
    ]

    all_passed = True
    for desc, result in checks:
        status = '✅ PASS' if result else '❌ FAIL'
        print(f'{status}  {desc}')
        if not result:
            all_passed = False

    print()
    if all_passed:
        print('🎉 All checks passed — safe to submit!')
    else:
        print('⚠️  Fix failing checks before submitting.')

except FileNotFoundError:
    print('submission.csv not found — run inference cell first.')

---
## Week 1 Complete ✅

### What you have now
- Full EDA: class distribution, duration stats, mel spectrogram visualisation
- Working audio pipeline: load → mel → resize → tensor
- EfficientNet-B0 + GeM pooling trained on fold 0
- OOF AUC tracked + per-class AUC histogram
- Valid `submission.csv` with sanity checks

### Week 2 targets
| Task | Change from Week 1 |
|---|---|
| Train all 5 folds | `cfg.TRAIN_FOLDS = [0,1,2,3,4]` |
| More epochs | `cfg.EPOCHS = 25` |
| Coarser second model | `N_FFT=2048, HOP=512, N_MELS=128` |
| Pseudo labels | Use soundscapes with confidence > 0.8 |
| Ensemble | Average predictions from 2+ models |
| Focal loss | Replace BCEWithLogitsLoss for rare classes |

> **Target OOF AUC:** ≥ 0.78 by end of Week 2